# Survival endpoints — OS and PFS

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

The spec (§2, §6) calls for TGI-metric → survival models for **both** overall survival (OS) and progression-free survival (PFS). Each tumor context now carries an OS link and a PFS link (parametric Weibull-PH driven by the week-8 relative tumor-size change); a simulation returns a curve per endpoint, and PFS is shorter than OS by construction.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
t = np.linspace(0, 156, 313)
tr = onkos.simulate(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0, t=t)
print('endpoints:', list(tr.survival))
print('median OS :', round(tr.median_os, 1), 'weeks')
print('median PFS:', round(tr.median_pfs, 1), 'weeks')
assert tr.median_pfs < tr.median_os

In [ ]:
plt.plot(t, tr.os_curve, label='OS')
plt.plot(t, tr.pfs_curve, '--', label='PFS')
plt.axhline(0.5, ls=':', color='grey'); plt.ylim(0, 1.02)
plt.xlabel('weeks'); plt.ylabel('survival fraction'); plt.legend(); plt.title('NSCLC OS vs PFS');
assert np.all(np.diff(tr.pfs_curve) <= 1e-9)  # monotone non-increasing

In [ ]:
# The virtual-trial divergence view reports both endpoints.
cmp = onkos.compare(ds, purpose='tgi', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0)
print('OS  divergence:', round(cmp.os_divergence, 3), '| median OS range :', cmp.median_os_range)
print('PFS divergence:', round(cmp.pfs_divergence, 3), '| median PFS range:', cmp.median_pfs_range)
assert cmp.pfs_divergence > 0

# Sensitivity can target PFS too.
res = onkos.sensitivity(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'},
                        target='median_pfs_weeks', n=200, seed=0)
print('drives PFS most:', res.dominant.symbol, f'({res.dominant.contribution*100:.0f}%)')